# Set up a single-HMS grid with a rerun, pinned to one version

<div class='alert alert-info'>

In this how-to you set up a grid of single hydrogen-rich stars
(single_HMS) with the physics of a POSYDON v2.0 rerun, and pin the
POSYDON inlists to one exact version, so the grid can be set up again
from the very same files later.

</div>

Four things decide what a grid runs, and each is pinned in its own way:

| What | Example | How you choose it |
|---|---|---|
| the release | `v2.0` | `[scenario] release` |
| the rerun | `matthias_no_age_limit` | `[scenario] rerun` |
| the version of the inlist files | commit `115de83` of POSYDON-MESA-INLISTS | the state of the folder `[scenario] inlist_repo` points at |
| the POSYDON code | `posydon` 2.2.x | the POSYDON you installed |

The release and rerun name the physics. The commit says which *version* of
those files you read: the v2.0 release is frozen, but the repository can
still change (fixes, new reruns), and a later `git pull` would silently
change a grid set up again from the same `ini` file. Pinning the commit
avoids that. All four end up in the grid's `setup_record.ini`.

<div class='alert alert-info'>

The outputs on this page were made with a stand-in for MESA, so the
compilation is instant, and the paths were shortened to `/home/user/...`.
With your own MESA, `posydon-grid setup` compiles MESA, which takes a few
minutes, and the paths are yours.

</div>

## Before you start

You need the same things as in
[Set up an HMS-HMS grid with a POSYDON rerun](hms_hms_rerun.ipynb#Before-you-start):

- POSYDON installed, with `posydon-grid` on your `PATH`;
- `MESA_DIR` and `MESASDK_ROOT` set, and the MESA SDK loaded, for the
  POSYDON version of MESA r11701;
- a clone of POSYDON-MESA-INLISTS on its `v2_sources` branch in
  `~/POSYDON-MESA-INLISTS`.

Make a new, empty folder for this grid and start Jupyter (or a terminal)
there. The cells below run in that folder; a line starting with `!` is a
shell command, which you can also type in a terminal without the `!`.

## 1. Pin the inlists to one commit

`posydon-grid` reads the folder `inlist_repo` as it is and never runs git
on it. So you pin a version by giving it a folder that is checked out at
that commit. A [git worktree](https://git-scm.com/docs/git-worktree)
does this without touching your main clone: it is a second folder of the
same repository, at another commit.

Find the commit you want (here, the newest one of `v2_sources`):

In [1]:
!git -C ~/POSYDON-MESA-INLISTS log --oneline -3

115de83 (HEAD -> v2_sources, origin/v2_sources, origin/HEAD) Remove the example ini files of r11701/running_scripts
48aff53 Remove releases/development until the next release is decided
fe9087b Say why mesa/default leaves out 15 lines of MESA's defaults


Make a folder of the repository at that commit:

In [2]:
!git -C ~/POSYDON-MESA-INLISTS worktree add --detach ~/POSYDON-MESA-INLISTS-115de83 115de83

Preparing worktree (detached HEAD 115de83)


HEAD is now at 115de83 Remove the example ini files of r11701/running_scripts


`~/POSYDON-MESA-INLISTS-115de83` now holds the files of commit `115de83`
and stays there, whatever you pull into `~/POSYDON-MESA-INLISTS`. (A
plain `git clone` followed by `git checkout 115de83` does the same.)

<details>
<summary><strong>And the POSYDON code?</strong></summary>

The POSYDON code writes the inlists, so it matters too. Install a
released version (`pip install posydon==<version>`) or check out a tag of
your POSYDON clone. The record notes the version, location and commit of
the POSYDON that set up the grid, and whether its code had local changes.

</details>

## 2. Choose the rerun

For single hydrogen-rich stars, the v2.0 grids used the rerun
`matthias_no_age_limit`. Its `scenario.ini` says so in `used_for`:

In [3]:
!grep -A1 used_for ~/POSYDON-MESA-INLISTS-115de83/r11701/releases/v2.0/reruns/matthias_no_age_limit/scenario.ini

used_for = CO-HMS_RLO (48), CO-HeMS (16), HMS-HMS (8), single_HMS (8)



## 3. Write the grid points

A single_HMS grid has one row per star: the metallicity (`initial_z`,
`Zbase`) and the initial mass in solar masses (`initial_mass`). Here are
five stars at Z = 0.0142:

In [4]:
import csv

masses = [10.0, 20.0, 30.0, 50.0, 80.0]
with open("grid.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["initial_z", "Zbase", "initial_mass"])
    for mass in masses:
        writer.writerow([0.0142, 0.0142, mass])

## 4. Write the ini file

Start from `posydon-grid init single_HMS single_HMS.ini`, or write it
directly. Note that `inlist_repo` points at the pinned folder, and that
the ZAMS model comes from it too. A single star is star 1, so its ZAMS
model is `zams_star1`, and `[output]` has only `star1`:

In [5]:
%%writefile single_HMS.ini
[grid]
type = single_HMS
mode = fixed
points = grid.csv
run_directory = run
zams_star1 = ~/POSYDON-MESA-INLISTS-115de83/r11701/ZAMS_models/zams_z1.42m2_y0.2703.data

[scenario]
release = v2.0
rerun = matthias_no_age_limit
inlist_repo = ~/POSYDON-MESA-INLISTS-115de83

[output]
history_interval = 1
star1 = history, final_model

[submission]
backend = slurm_array
partition = my_partition
account = my_account
walltime = 2-00:00:00
cpus_per_task = 4

Writing single_HMS.ini


## 5. Check and set up

The rerun removes the age limit. `posydon-grid explain` shows it:

In [6]:
!posydon-grid explain single_HMS.ini > explain.txt
!grep '^[a-z]' explain.txt
!grep -A2 '^  max_age ' explain.txt

star1_formation_step0_controls
star1_formation_step0_job


  max_age = -1  [posydon_inlists] r11701/releases/v2.0/reruns/matthias_no_age_limit/common/binary/inlist1
      over 1.4e10  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist1
      over 1d36  [mesa_inlists] r11701/mesa/default/defaults/star/controls.defaults


POSYDON runs single stars with MESA's single-star executable, as the
*formation steps* of star 1: here one step, `star1_formation_step0`,
which runs the star from ZAMS to its end. (A single_HeMS grid has several
steps, see [Set up a single-HeMS grid](single_hems_rerun.ipynb).) The
star's inlists come from the release's star-1 inlist of the binary grids
(`common/binary/inlist1`), with the rerun's changes and the release's
`single_star_flag` on top.

Set up the run directory:

In [7]:
!posydon-grid setup single_HMS.ini
!ls run run/star1

run:
binary                      run_grid.sh
cleanup.slurm               setup_record.ini
column_lists                star1
job_array_grid_submit.slurm star2
mk

run/star1:
inlist_step0 make         src


The `[output]` options went into the step that runs the star: it saves
the final model as `final_star1.mod`, which the post-processing reads.

In [8]:
!grep -n "save_model_when_terminate\|save_model_filename" run/star1/inlist_step0

1681:	save_model_when_terminate = .true.
1682:	save_model_filename = 'final_star1.mod'


## 6. Check the version in the record

The record names the folder that was read, its commit, and whether its
files differed from that commit:

In [9]:
!sed -n '/^\[scenario\]/,/^$/p' run/setup_record.ini
!sed -n '/^\[posydon\]/,/^$/p' run/setup_record.ini

[scenario]
release = v2.0
rerun = matthias_no_age_limit
mesa_version = r11701
mesa = default
inlist_repo = /home/user/POSYDON-MESA-INLISTS-115de83
inlist_remote = https://github.com/POSYDON-code/POSYDON-MESA-INLISTS.git
inlist_commit = 115de83c33bad5013cb09837fcc679d1d45e816b
inlist_dirty = False
inlist_fingerprint = b287370f6b57cd2110ddeaf26b1f8e467d4a4f71bb62a5cc4b23389922f24c87
inlist_files = 15



[posydon]
version = 2.2.4+3.ga271efb93
location = /home/user/POSYDON
commit = e2d6bad701187b9a473f339004732d4c5cc215a6
dirty = False
python = 3.11.9 (/home/user/miniconda3/envs/posydon/bin/python)



To set up this grid again later, from the same files, keep the `ini` file
and the folder `~/POSYDON-MESA-INLISTS-115de83`. If the folder is gone,
make it again at the `inlist_commit` of the record (step 1): the
`inlist_fingerprint` of the new record is then the same.

Submit the grid as in
[the HMS-HMS how-to](hms_hms_rerun.ipynb#6.-Submit-the-grid):
`cd run && ./run_grid.sh`.

## See also

- [Set up a single-HeMS grid in the same way](single_hems_rerun.ipynb)
- [The ini file](../../../components-overview/mesa_grids/inifile.rst)